<a href="https://colab.research.google.com/github/UPM-RSTI/OpenSSL---PKI/blob/main/Practica_PKI_OpenSSL_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Para ejecutar la práctica:** pulse el botón *Open in Colab*, después *Archivo → Guardar una copia en Drive* para conservar sus respuestas, y ejecute las celdas en orden.

# Práctica PKI con OpenSSL: firma digital, CA, TLS y OCSP

1. **Parte 1.** Par de claves RSA: firma y verificación de un fichero.
2. **Parte 2.** CA propia que emite el certificado de un servidor web HTTPS (Apache).
3. **Parte 3.** Revocación de certificados y servicio OCSP.

**Cómo usar este notebook**

- Cada celda contiene los comandos tal como se escribirían en una terminal Linux. Ejecútelas **en orden** (`Mayús + Intro`) y lea la salida.
- Cada celda empieza con `cd` a la carpeta de trabajo, como si abriera la terminal en ella.
- Donde aparezca `pass:practica`, es la contraseña de la clave. En una terminal normal OpenSSL la pediría por teclado; en Colab se escribe en el comando. Puede cambiarla, pero use la misma en todos los pasos.

**Entrega.** Responda en el cuestionario de Moodle a las preguntas numeradas: **(¿?)** respuesta escrita, **(Adjuntar)** captura de pantalla de la salida de la celda indicada, **(Test)** opción múltiple.

## Preparación
Instalación de OpenSSL y Apache, y creación de la carpeta de trabajo.

In [ ]:
%%bash
exec 2>&1   # mensajes y errores en orden, como en la terminal
echo "### Instalando paquetes (tarda ~1 minuto)..."
apt-get update > /dev/null
apt-get install -y openssl apache2 curl > /dev/null
openssl version
mkdir -p /content/practica-pki
echo "### Carpeta de trabajo creada: /content/practica-pki"

---
# Parte 1. Par de claves y firma digital

### 1.1 Generación de la clave privada
`clavePri.pem` es el fichero que contendrá la clave privada y 4096 su tamaño en bits.

In [ ]:
%%bash
exec 2>&1   # mensajes y errores en orden, como en la terminal
cd /content/practica-pki
openssl genrsa -out clavePri.pem 4096
echo "### Clave privada generada:"
ls -l clavePri.pem

### 1.2 Generación de la clave pública
`clavePub.pem` contendrá la clave pública en el formato indicado con `-outform` (PEM).

In [ ]:
%%bash
exec 2>&1   # mensajes y errores en orden, como en la terminal
cd /content/practica-pki
openssl rsa -in clavePri.pem -out clavePub.pem -outform PEM -pubout
echo "===== clavePri.pem ====="
cat clavePri.pem
echo "===== clavePub.pem ====="
cat clavePub.pem

**Pregunta 1 (¿?)** Copie la primera y la última línea de `clavePri.pem` y de `clavePub.pem`. ¿Qué indican esas cabeceras?

**Respuesta:** _(escriba aquí su respuesta o péguela en el cuestionario de Moodle)_

**Pregunta 2 (¿?)** ¿Es correcto que la clave privada sea legible? Justifique la respuesta.

**Respuesta:** _(escriba aquí su respuesta o péguela en el cuestionario de Moodle)_

**Pregunta 3 (Adjuntar)** Adjunte una captura de las dos celdas anteriores en la que se vean ambas claves y los comandos con los que las generó.

### 1.3 Cifrado de la clave privada
`aes256` es el algoritmo de cifrado simétrico y `clavePriCif.pem` el fichero con la clave privada cifrada.

In [ ]:
%%bash
exec 2>&1   # mensajes y errores en orden, como en la terminal
cd /content/practica-pki
openssl rsa -in clavePri.pem -aes256 -out clavePriCif.pem -passout pass:practica
echo "===== clavePriCif.pem ====="
cat clavePriCif.pem

**Pregunta 4 (¿?)** Compare `clavePriCif.pem` con `clavePri.pem`. ¿Es ahora legible la clave privada? ¿Qué necesita un atacante que robe `clavePriCif.pem` para poder usarla?

**Respuesta:** _(escriba aquí su respuesta o péguela en el cuestionario de Moodle)_

### 1.4 Fichero a firmar
**Cambie el texto entre comillas por su nombre y apellidos completos** antes de ejecutar la celda.

In [ ]:
%%bash
exec 2>&1   # mensajes y errores en orden, como en la terminal
cd /content/practica-pki
echo "Nombre Apellido1 Apellido2" > archivoEntrada.txt
echo "### Contenido de archivoEntrada.txt:"
cat archivoEntrada.txt

### 1.5 Firma del fichero
(A) con la clave privada en claro y (B) con la clave privada cifrada. `sha256` es el algoritmo de resumen y `firma`/`firma2` el resultado.

In [ ]:
%%bash
exec 2>&1   # mensajes y errores en orden, como en la terminal
cd /content/practica-pki
echo "### (A) Firma con la clave privada en claro"
openssl dgst -sha256 -sign clavePri.pem -out firma archivoEntrada.txt
echo "### (B) Firma con la clave privada cifrada"
openssl dgst -sha256 -sign clavePriCif.pem -passin pass:practica -out firma2 archivoEntrada.txt
ls -l firma firma2

### 1.6 Verificación de las firmas
El resultado debe ser `Verified OK`.

In [ ]:
%%bash
exec 2>&1   # mensajes y errores en orden, como en la terminal
cd /content/practica-pki
echo "### (A) Verificación de firma"
openssl dgst -sha256 -verify clavePub.pem -signature firma archivoEntrada.txt
echo "### (B) Verificación de firma2"
openssl dgst -sha256 -verify clavePub.pem -signature firma2 archivoEntrada.txt
echo "### ¿Son iguales firma y firma2?"
cmp firma firma2 && echo "Sí, son idénticas"

¿Qué ocurre si se modifica el fichero después de firmarlo? Se prueba con una copia modificada:

In [ ]:
%%bash
exec 2>&1   # mensajes y errores en orden, como en la terminal
cd /content/practica-pki
cp archivoEntrada.txt archivoModificado.txt
echo "cambio" >> archivoModificado.txt
echo "### Verificación de firma con el fichero modificado"
openssl dgst -sha256 -verify clavePub.pem -signature firma archivoModificado.txt
echo "### Fin de la prueba"

**Pregunta 5 (Adjuntar)** Adjunte una captura de las dos celdas anteriores (las dos firmas verificadas y el fallo con el fichero modificado).

**Pregunta 6 (¿?)** ¿Se obtiene `Verified OK` en las dos variantes A y B? ¿Por qué? ¿Por qué `firma` y `firma2` son idénticas?

**Respuesta:** _(escriba aquí su respuesta o péguela en el cuestionario de Moodle)_

**Pregunta 7 (Test)** ¿Qué archivo(s) debe enviar a otra persona para que pueda verificar la firma, suponiendo que ya tiene el fichero original? *(Responda en Moodle: Clave pública / Clave privada / Ninguno / Firma)*

**Pregunta 8 (¿?)** ¿Qué debería hacerse con `clavePri.pem` una vez generado `clavePriCif.pem`? Justifique la respuesta.

**Respuesta:** _(escriba aquí su respuesta o péguela en el cuestionario de Moodle)_

---
# De la clave pública al certificado
En la Parte 1, quien verifica confía en que `clavePub.pem` es realmente suya, pero nada en ese fichero lo demuestra. Un **certificado X.509** une una clave pública con una identidad y lo firma una **Autoridad de Certificación (CA)** de confianza. En las Partes 2 y 3 se crea una CA propia autofirmada; OpenSSL guarda su estado en el directorio `demoCA`.

**Pregunta 9 (¿?)** Si envía `clavePub.pem` por correo, ¿cómo puede el receptor estar seguro de que es suya? ¿Qué aporta una CA en este escenario?

**Respuesta:** _(escriba aquí su respuesta o péguela en el cuestionario de Moodle)_

---
# Parte 2. CA propia y servidor web con TLS

### 2.1 Descarga de los scripts
Se descarga y descomprime `opensslscripts.tar`, que crea el directorio `container` con los scripts (`testca`, `miCA.sh`) y las configuraciones (`CAss.cnf`, `openssl.cnf`, `website.cnf`, `openssl_ocsp.cnf`).

In [ ]:
%%bash
exec 2>&1   # mensajes y errores en orden, como en la terminal
cd /content/practica-pki
wget -q -O opensslscripts.tar "https://drive.upm.es/s/4jgtCCDfpFXeRkX/download"
tar xvf opensslscripts.tar
echo "### Contenido de container:"
ls -l container

### 2.2 Ajuste de las configuraciones
Antes de usarlas se ajustan con `sed`:
1. La IP de ejemplo (`172.25.28.235`) se sustituye por la IP de esta máquina.
2. El certificado de la CA se genera con la extensión `CA:TRUE` (sección `[ v3 ]` de `CAss.cnf`).
3. Los certificados del servidor se emiten como `CA:FALSE` y con uso `server`, para que los clientes TLS los acepten.
4. Se borra `log.txt`, un registro de una ejecución anterior, y se copia `.rnd` (semilla aleatoria) al directorio personal, donde la busca `openssl.cnf`.

In [ ]:
%%bash
exec 2>&1   # mensajes y errores en orden, como en la terminal
cd /content/practica-pki/container
IP=$(hostname -I | awk '{print $1}')
echo "IP de la máquina: $IP"
sed -i "s/172.25.28.235/$IP/g" website.cnf openssl_ocsp.cnf
sed -i "s/^default_policy.*/&\nx509_extensions = v3/" CAss.cnf
sed -i "s/^basicConstraints=CA:TRUE/basicConstraints=CA:FALSE/" openssl.cnf
sed -i "s/^nsCertType = client, email, objsign/nsCertType = server/" openssl.cnf openssl_ocsp.cnf
rm -f log.txt
cp .rnd ~/.rnd
echo "### Líneas modificadas:"
grep -n "$IP" website.cnf openssl_ocsp.cnf
grep -n "x509_extensions" CAss.cnf
grep -n -E "^basicConstraints=|^nsCertType" openssl.cnf openssl_ocsp.cnf

### 2.3 Creación de la CA
Se usa el fichero de configuración `CAss.cnf` a través del script `testca`.

In [ ]:
%%bash
exec 2>&1   # mensajes y errores en orden, como en la terminal
cd /content/practica-pki/container
sh testca -ca < /dev/null
echo "### Directorios y ficheros creados:"
ls -lRt demoCA

### 2.4 Clave privada y certificado de la CA

In [ ]:
%%bash
exec 2>&1   # mensajes y errores en orden, como en la terminal
cd /content/practica-pki/container
openssl rsa -in demoCA/private/cakey.pem -text -noout

Por defecto OpenSSL genera claves RSA; también puede usar curvas elípticas (`ecparam`) o parámetros Diffie-Hellman (`dhparam`).

In [ ]:
%%bash
exec 2>&1   # mensajes y errores en orden, como en la terminal
cd /content/practica-pki/container
openssl x509 -in demoCA/cacert.pem -text -noout

**Pregunta 10 (¿?)** ¿Identifica la salida de `openssl rsa -text`? Relacione cada campo (`modulus`, `publicExponent`, `privateExponent`, `prime1`, `prime2`…) con los elementos del algoritmo RSA. ¿Está cifrada la clave privada de la CA? ¿Es adecuado?

**Respuesta:** _(escriba aquí su respuesta o péguela en el cuestionario de Moodle)_

**Pregunta 11 (¿?)** En el certificado de la CA, ¿qué tienen en común los campos `Issuer` y `Subject`? ¿Qué indica la extensión `Basic Constraints: CA:TRUE`?

**Respuesta:** _(escriba aquí su respuesta o péguela en el cuestionario de Moodle)_

### 2.5 Petición de certificado para el servidor web
Con la configuración de `website.cnf` se genera la clave privada del servidor (`website.key`) y la petición (`website_req.pem`), con la IP como nombre alternativo.

In [ ]:
%%bash
exec 2>&1   # mensajes y errores en orden, como en la terminal
cd /content/practica-pki/container
IP=$(hostname -I | awk '{print $1}')
echo "IP de la máquina: $IP"
openssl req -batch -config website.cnf -new -keyout website.key -out website_req.pem -addext "subjectAltName=IP:$IP"
echo "### Contenido de la petición:"
openssl req -in website_req.pem -text -noout

### 2.6 Emisión del certificado por la CA
`-batch` responde automáticamente `y` a las preguntas de confirmación.

In [ ]:
%%bash
exec 2>&1   # mensajes y errores en orden, como en la terminal
cd /content/practica-pki/container
openssl ca -batch -config openssl.cnf -verbose -cert demoCA/cacert.pem -policy policy_match -out website.pem -infiles website_req.pem

In [ ]:
%%bash
exec 2>&1   # mensajes y errores en orden, como en la terminal
cd /content/practica-pki/container
echo "### Certificado emitido:"
openssl x509 -in website.pem -text -noout
echo "### Verificación con el certificado de la CA:"
openssl verify -CAfile demoCA/cacert.pem website.pem
echo "### Lista de certificados de la CA (index.txt):"
cat demoCA/index.txt

**Pregunta 12 (¿?)** ¿Qué diferencias hay entre la petición (`website_req.pem`) y el certificado (`website.pem`)? ¿Quién firma cada uno? Explique el significado de cada columna de la línea de `index.txt`.

**Respuesta:** _(escriba aquí su respuesta o péguela en el cuestionario de Moodle)_

### 2.7 Configuración de Apache con TLS
- `a2enmod ssl` crea los enlaces de `mods-available` a `mods-enabled` (`ssl.load`, `ssl.conf`, `socache_shmcb.load`).
- `a2ensite default-ssl` enlaza `sites-available/default-ssl.conf` en `sites-enabled`.
- `sed` sustituye en `default-ssl.conf` los certificados de ejemplo (*snakeoil*) por los nuestros.

In [ ]:
%%bash
exec 2>&1   # mensajes y errores en orden, como en la terminal
a2enmod ssl
a2ensite default-ssl
echo "ServerName localhost" > /etc/apache2/conf-enabled/servername.conf
sed -i "s#/etc/ssl/certs/ssl-cert-snakeoil.pem#/content/practica-pki/container/website.pem#" /etc/apache2/sites-available/default-ssl.conf
sed -i "s#/etc/ssl/private/ssl-cert-snakeoil.key#/content/practica-pki/container/website.key#" /etc/apache2/sites-available/default-ssl.conf
echo "### Directivas SSL en default-ssl.conf:"
grep "SSLCertificate" /etc/apache2/sites-available/default-ssl.conf
echo "### Arrancando Apache:"
apache2ctl restart
echo "Apache en marcha"

### 2.8 Acceso por HTTPS
En Colab no se puede abrir la página con el navegador, así que se usa `curl`. Sin indicar la CA equivale a un navegador que no la conoce; con `--cacert` equivale a haberla importado como autoridad raíz en el navegador.

In [ ]:
%%bash
exec 2>&1   # mensajes y errores en orden, como en la terminal
cd /content/practica-pki/container
IP=$(hostname -I | awk '{print $1}')
echo "IP de la máquina: $IP"
echo "### 1) Sin confiar en nuestra CA:"
curl -sS https://$IP/
echo
echo "### 2) Confiando en nuestra CA:"
curl -sS --cacert demoCA/cacert.pem https://$IP/ | grep -m1 -o "It works"

In [ ]:
%%bash
exec 2>&1   # mensajes y errores en orden, como en la terminal
cd /content/practica-pki/container
IP=$(hostname -I | awk '{print $1}')
echo "IP de la máquina: $IP"
echo "### Conexión TLS: certificado del servidor, emisor y verificación"
openssl s_client -connect $IP:443 -CAfile demoCA/cacert.pem < /dev/null | grep -E "s:|i:|Verify return code"

> En un Linux propio con navegador: importe `demoCA/cacert.pem` como autoridad raíz (Chrome: *Configuración → Privacidad y seguridad → Seguridad → Gestionar certificados*; Firefox: *Ajustes → Privacidad y seguridad → Ver certificados → Autoridades → Importar*) y abra `https://` seguido de la IP de su máquina.

**Pregunta 13 (Adjuntar)** Adjunte una captura de las dos celdas anteriores.

Acceso con un nombre distinto del que figura en el certificado:

In [ ]:
%%bash
exec 2>&1   # mensajes y errores en orden, como en la terminal
cd /content/practica-pki/container
echo "### Acceso a https://localhost"
curl -sS --cacert demoCA/cacert.pem https://localhost/
echo "### Fin de la prueba"

**Pregunta 14 (¿?)** ¿Por qué falla la conexión a `https://localhost` si el servidor y la CA son los mismos? ¿Para qué sirve la extensión `subjectAltName` del certificado?

**Respuesta:** _(escriba aquí su respuesta o péguela en el cuestionario de Moodle)_

---
# Parte 3. Revocación y servicio OCSP

Un certificado puede dejar de ser fiable antes de caducar (por ejemplo, si roban su clave). La CA lo **revoca** y un servicio **OCSP** responde en línea si un certificado es válido (`good`), está revocado (`revoked`) o es desconocido (`unknown`).

### 3.1 Revocación del certificado anterior
La CA no permite dos certificados válidos con el mismo nombre. Se guarda una copia del certificado actual y se revoca; en `index.txt` su línea pasa de `V` a `R`.

In [ ]:
%%bash
exec 2>&1   # mensajes y errores en orden, como en la terminal
cd /content/practica-pki/container
cp website.pem website_revocado.pem
openssl ca -config openssl.cnf -revoke website_revocado.pem
echo "### index.txt:"
cat demoCA/index.txt

### 3.2 Certificado del responder OCSP
Se crea la solicitud del certificado del responder y se le pide a la CA que la firme.

In [ ]:
%%bash
exec 2>&1   # mensajes y errores en orden, como en la terminal
cd /content/practica-pki/container
openssl req -batch -new -nodes -out ocsp.csr -keyout ocsp.key -config openssl_ocsp.cnf -extensions v3_OCSP
openssl ca -batch -in ocsp.csr -out ocsp.pem -config openssl_ocsp.cnf -extensions v3_OCSP -policy policy_anything
echo "### Certificado del responder:"
openssl x509 -in ocsp.pem -noout -subject -ext extendedKeyUsage

### 3.3 Nuevo certificado del servidor con la dirección del OCSP
Se pide un nuevo certificado y la CA lo firma con `openssl_ocsp.cnf`, que añade la URL del responder. Después se reinicia Apache para que use el nuevo certificado.

In [ ]:
%%bash
exec 2>&1   # mensajes y errores en orden, como en la terminal
cd /content/practica-pki/container
IP=$(hostname -I | awk '{print $1}')
echo "IP de la máquina: $IP"
openssl req -batch -config website.cnf -new -keyout website.key -out website_req.pem -addext "subjectAltName=IP:$IP"
openssl ca -batch -config openssl_ocsp.cnf -verbose -cert demoCA/cacert.pem -policy policy_match -out website.pem -infiles website_req.pem
echo "### Extensión con la dirección del OCSP:"
openssl x509 -in website.pem -noout -ext authorityInfoAccess
echo "### index.txt:"
cat demoCA/index.txt
apache2ctl restart
echo "Apache reiniciado con el nuevo certificado"

### 3.4 Arranque del responder OCSP
Se lanza en segundo plano (`&`) escuchando en el puerto 8888; su registro queda en `log.txt`.

In [ ]:
%%bash
exec 2>&1   # mensajes y errores en orden, como en la terminal
cd /content/practica-pki/container
nohup openssl ocsp -index demoCA/index.txt -port 8888 -rsigner ocsp.pem -rkey ocsp.key -CA demoCA/cacert.pem -text -out log.txt > /dev/null 2>&1 &
sleep 2
echo "### Proceso del responder OCSP:"
pgrep -a -f "openssl ocsp -index"

### 3.5 Consulta del estado de los certificados
Resultado esperado: `Response verify OK`, `website.pem: good` y `website_revocado.pem: revoked`.

In [ ]:
%%bash
exec 2>&1   # mensajes y errores en orden, como en la terminal
cd /content/practica-pki/container
IP=$(hostname -I | awk '{print $1}')
echo "IP de la máquina: $IP"
echo "### Certificado actual"
openssl ocsp -CAfile demoCA/cacert.pem -issuer demoCA/cacert.pem -cert website.pem -url http://$IP:8888
echo
echo "### Certificado revocado"
openssl ocsp -CAfile demoCA/cacert.pem -issuer demoCA/cacert.pem -cert website_revocado.pem -url http://$IP:8888

Registro del responder:

In [ ]:
%%bash
exec 2>&1   # mensajes y errores en orden, como en la terminal
cd /content/practica-pki/container
grep -E "Serial Number: [0-9A-F]+$|Cert Status|Revocation Time" log.txt

**Pregunta 15 (Adjuntar)** Adjunte capturas de la celda 3.3 (extensión *Authority Information Access*) y de las dos consultas OCSP.

**Pregunta 16 (¿?)** ¿Qué significa `Response verify OK`? ¿Qué certificado y qué clave firman la respuesta OCSP, y por qué el cliente puede confiar en ella?

**Respuesta:** _(escriba aquí su respuesta o péguela en el cuestionario de Moodle)_

**Pregunta 17 (¿?)** ¿Qué ventajas tiene OCSP frente a las listas de revocación (CRL)? ¿Qué información revela al responder sobre la navegación del usuario?

**Respuesta:** _(escriba aquí su respuesta o péguela en el cuestionario de Moodle)_

---
## Final de la práctica
Se paran el responder OCSP y Apache.

In [ ]:
%%bash
exec 2>&1   # mensajes y errores en orden, como en la terminal
pkill -f "openssl ocsp -index"
apache2ctl stop
echo "Servicios parados"